# 月データ探索ツール（発展編）

`explore.ipynb`（標準編）で「気になる関係」を見つけた人向けに、統計的検定・回帰・疑似相関の
検証まで踏み込むノートブックです。情報Ⅰの範囲を超える内容（`scipy.stats`・`numpy.polyfit`・
`seaborn`）を使うため、標準編とは別ファイルに分けてあります。

対象：数学や地学基礎で相関・回帰・検定に触れたことがある人、SSH／理数科の探究活動など。

構成：
- **A. べき乗則フィット**：クレーターの直径分布は「べき乗則」に従うことが知られています。
  両対数グラフで直線になるかどうかを確認し、傾き（べき指数）を回帰で求めます。
- **B. 月齢と地球環境の相関検定**：「月齢と地震には関係がある」という言説を、実際にデータで
  検証します。相関係数だけでなく統計的検定（t検定）も行い、結果の解釈に注意すべき点を学びます。

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB and not os.path.exists('data'):
    print("\u26a0\ufe0f dataフォルダが見つかりません。explore.ipynbの案内を参照してリポジトリごと取得してください。")

def _read(name):
    for p in (f'../data/{name}', f'data/{name}'):
        if os.path.exists(p):
            return pd.read_csv(p)
    raise FileNotFoundError(name)

craters = _read('craters_subset.csv')
moon_earth = _read('moon_earth_correlation.csv')
sns.set_theme(style='whitegrid')
print(f'craters: {len(craters):,}件 / moon_earth_correlation: {len(moon_earth):,}件')

## A. クレーター直径のべき乗則フィット

「直径Dより大きいクレーターの個数 N(>D)」を数え、横軸・縦軸をともに対数にして散布図を描くと、
多くの天体でおおむね直線（＝べき乗則 $N(>D) \propto D^{-b}$）になることが知られています。
`numpy.polyfit` で対数-対数の傾き（べき指数 $b$）を最小二乗法で求めてみましょう。

In [ ]:
diam = np.sort(craters['diam_km'].to_numpy())[::-1]
n_cumulative = np.arange(1, len(diam) + 1)  # diam[i]より大きい(以上の)クレーターの個数

log_d = np.log10(diam)
log_n = np.log10(n_cumulative)

slope, intercept = np.polyfit(log_d, log_n, 1)
fit_n = 10 ** (slope * log_d + intercept)

fig, ax = plt.subplots(figsize=(7, 6))
ax.loglog(diam, n_cumulative, '.', markersize=3, alpha=0.4, label='実データ')
ax.loglog(diam, fit_n, 'r-', linewidth=2, label=f'べき乗則フィット（傾き b = {slope:.2f}）')
ax.set_xlabel('直径 D [km]（対数）')
ax.set_ylabel('直径D以上のクレーター累積個数 N(>D)（対数）')
ax.set_title('クレーターサイズ頻度分布（累積）')
ax.legend()
plt.tight_layout()
plt.show()

print(f'べき指数 b \u2248 {abs(slope):.2f}（一般的な文献値はおよそ2前後）')

## B. 月齢と地球環境の相関検定（反面教師）

「月齢（満ち欠け）は地震を引き起こす」という話を聞いたことがあるかもしれません。
ここでは、実際にデータで検証します。

使うデータ（`data/moon_earth_correlation.csv`）：
- `distance_km`：地球ー月の距離（JPL HORIZONSより算出）
- `theoretical_tide_force_index`：距離から計算した理論潮汐力の指数（距離の3乗に反比例。
  実測データではなく、教材側で計算式から求めた値）
- `earthquake_count_m4_plus`：全世界のマグニチュード4.0以上の地震の日次件数（USGS）

**気象庁の実測潮位データは使っていません**（過去5年分を機械可読形式で一括取得できる無料の手段が
存在しなかったため）。その代わり、距離から理論的に計算できる潮汐力の指数を使っています。

In [ ]:
def interpret_correlation(label, r, p, n):
    print(f'[{label}] n={n}, r={r:.4f}, p={p:.4g}')
    if abs(r) > 0.7:
        print('  \u2192 強い相関。ただし、これは計算式で直接結びついているだけかもしれません。')
        print('    「AとBに強い相関がある」＝「Aの値からBの値がほぼ計算できる」という意味で、')
        print('    因果関係を示すとは限らない点に注意してください。')
    elif p < 0.05:
        print('  \u2192 統計的には「偶然とは考えにくい」相関ですが、相関の大きさ(|r|)はとても小さいです。')
        print('    サンプル数が多いと、意味のある関係が無くてもp値は小さくなりがちです。')
        print('    |r|の値そのものを見て、実質的に意味のある関係かどうかを判断しましょう。')
    else:
        print('  \u2192 相関係数は小さく、統計的にも「相関が無い」という帰無仮説を棄却できません。')
        print('    第三の要因（季節、観測網の変化など）を考える前に、まず「関係が無いらしい」')
        print('    ということ自体が結論になり得ます。')
    print()


r1, p1 = stats.pearsonr(moon_earth['distance_km'], moon_earth['theoretical_tide_force_index'])
interpret_correlation('距離 vs 理論潮汐力指数（計算式で直結している比較用の例）', r1, p1, len(moon_earth))

r2, p2 = stats.pearsonr(moon_earth['theoretical_tide_force_index'], moon_earth['earthquake_count_m4_plus'])
interpret_correlation('理論潮汐力指数 vs 地震件数(M4.0以上)', r2, p2, len(moon_earth))

r3, p3 = stats.pearsonr(moon_earth['moon_phase'], moon_earth['earthquake_count_m4_plus'])
interpret_correlation('月相(輝面比) vs 地震件数(M4.0以上)', r3, p3, len(moon_earth))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(moon_earth['distance_km'], moon_earth['theoretical_tide_force_index'], s=6, alpha=0.4)
axes[0].set_xlabel('地球ー月の距離 [km]')
axes[0].set_ylabel('理論潮汐力指数')
axes[0].set_title(f'比較用：計算式で直結した関係 (r={r1:.3f})')

axes[1].scatter(moon_earth['theoretical_tide_force_index'], moon_earth['earthquake_count_m4_plus'], s=6, alpha=0.4)
axes[1].set_xlabel('理論潮汐力指数')
axes[1].set_ylabel('地震件数(M4.0以上/日)')
axes[1].set_title(f'本題：潮汐力 vs 地震件数 (r={r2:.3f})')

plt.tight_layout()
plt.show()

### t検定：「大潮に近い日」と「そうでない日」で地震件数の平均に差はあるか

理論潮汐力指数が上位25%の日（大潮に近い＝潮汐力が強い）と、下位25%の日（小潮に近い）を比べ、
地震件数の平均に統計的な差があるかをt検定で調べます。

In [ ]:
q75 = moon_earth['theoretical_tide_force_index'].quantile(0.75)
q25 = moon_earth['theoretical_tide_force_index'].quantile(0.25)

high_tide_days = moon_earth[moon_earth['theoretical_tide_force_index'] >= q75]['earthquake_count_m4_plus']
low_tide_days = moon_earth[moon_earth['theoretical_tide_force_index'] <= q25]['earthquake_count_m4_plus']

t_stat, p_value = stats.ttest_ind(high_tide_days, low_tide_days, equal_var=False)

print(f'潮汐力 上位25%の日: 平均 {high_tide_days.mean():.2f} 件/日 (n={len(high_tide_days)})')
print(f'潮汐力 下位25%の日: 平均 {low_tide_days.mean():.2f} 件/日 (n={len(low_tide_days)})')
print(f't = {t_stat:.3f}, p = {p_value:.4f}')
if p_value < 0.05:
    print('\u2192 p < 0.05：統計的には「平均に差が無い」とは言い切れない結果です。')
    print('   ただし平均の差そのものの大きさも確認し、実質的な意味があるか考えましょう。')
else:
    print('\u2192 p \u2265 0.05：この検定では「平均に差がある」とは言えません。')
    print('   これは「月齢と地震に関係が無い」という一般的な理解と整合的な結果です。')

## まとめ：相関・検定を読むときの注意点

1. **相関係数 r の大きさ**を必ず見る（p値が小さくても|r|が小さければ実質的な関係は弱い）
2. **サンプル数が多いと、弱い関係でも統計的に「有意」になりやすい**（今回のデータは約1,800日分）
3. **強い相関＝因果関係ではない**。特に、片方がもう片方の計算式そのものになっている場合（今回の
   「距離と理論潮汐力指数」のように）は、相関が強くて当然
4. データが実測ではなく計算で求めた値を含む場合（今回の理論潮汐力指数など）、それがどこまで
   現実を近似できているかを意識する

## データの出典

- USGS Astrogeology（Robbins Crater DB）、NASA JPL HORIZONS System（地球ー月の距離・位相角）、
  USGS FDSN Event Web Service（地震データ、M4.0以上、全世界）。詳細は`explore.ipynb`および
  `docs/requirements_v2.1.md`を参照。
- 理論潮汐力指数は、平均距離384,400kmを基準に (384400 / distance_km)^3 として教材側で算出した値。